<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/worktree-ainaval-review/Homework_Solutions/NB04_Homework_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB04 Homework Solutions

Answers to the "Homework / Practice Ideas" section of
`NB04_NumPy_for_Engineering_Computation_Vectors_Matrices_and_Linear_Algebra.ipynb`:

1. Extend Section 5's mooring problem to 3 lines (3 unknowns) with a moment-balance equation.
2. Make Section 5's system singular on purpose and observe `np.linalg.det` / `np.linalg.solve`.
3. Use `numpy.linalg.lstsq` on an over-determined 3-line mooring system.
4. Repeat Section 6's eigenvector computation on two uncorrelated Sonar bands.
5. Broadcast a (4,) per-timestamp correction array against Section 3's `readings` array.


## Homework 1 — Extending the mooring problem to 3 lines with a moment-balance equation

The class's Section 5 problem: a vessel held by 2 mooring lines at fixed angles,
resisting one environmental force, solved for line tensions `T1, T2` with
`np.linalg.solve`. I extend this to 3 mooring lines (3 unknowns `T1, T2, T3`),
which needs a 3rd independent equation. I add force-equilibrium in x and y
(2 equations) plus a moment-balance equation about the vessel's center of
gravity (1 equation) -- 3 equations for 3 unknowns, same well-posed structure
as the class example, just extended.

Each line attaches to the hull at a known offset from the center of gravity
(CG); its moment contribution is `r x F` (2-D cross product = scalar:
`r_x * F_y - r_y * F_x`).


In [1]:
import numpy as np

# Three mooring lines, each with an angle (deg, measured like Section 5) and an
# attachment point relative to the vessel's center of gravity (x, y in meters)
angles_deg = np.array([30.0, 150.0, 250.0])
attach_points = np.array([
    [80.0,  5.0],   # line 1: near the bow, slightly to starboard
    [80.0, -5.0],   # line 2: near the bow, slightly to port
    [-90.0, 0.0],   # line 3: at the stern, centered
])

env_force_kN, env_angle_deg = 40.0, 200.0
a = np.radians(angles_deg)
ae = np.radians(env_angle_deg)

# Force-equilibrium rows (x, y), same structure as Section 5
Fx_row = np.cos(a)
Fy_row = np.sin(a)

# Moment row: unit force direction per line, moment arm = r_x*Fy_unit - r_y*Fx_unit
moment_row = attach_points[:, 0] * np.sin(a) - attach_points[:, 1] * np.cos(a)

A3 = np.vstack([Fx_row, Fy_row, moment_row])
b3 = np.array([
    -env_force_kN * np.cos(ae),
    -env_force_kN * np.sin(ae),
    0.0,   # assume the environmental force acts through the CG -> no external moment
])

T1, T2, T3 = np.linalg.solve(A3, b3)
print(f"Line 1 tension: {T1:.2f} kN")
print(f"Line 2 tension: {T2:.2f} kN")
print(f"Line 3 tension: {T3:.2f} kN")

residual = A3 @ np.array([T1, T2, T3]) - b3
print("\nResidual (should be ~0):", residual)
print("det(A3):", np.linalg.det(A3))


Line 1 tension: 28.06 kN
Line 2 tension: -12.80 kN
Line 3 tension: -6.44 kN

Residual (should be ~0): [7.10542736e-15 3.55271368e-15 3.14997195e-14]
det(A3): 131.29791117349913


Same honesty check the class applied to the 2-line case: the residual is
effectively zero, so the solve is numerically correct, and `det(A3)` is far
from zero so this specific 3x3 system has one unique solution. As in the
2-line case, at least one tension can come out negative (a line "pushing" is
not physically real for a mooring line, which can only pull) -- if that
happens here it would mean this particular arrangement of 3 lines is not
adequate for this force, not that the math is wrong.


## Homework 2 — Making Section 5's system singular on purpose

Setting both line angles equal removes any real difference in pulling
direction between the two lines, so the system should become singular
(non-invertible).


In [2]:
angle1_deg, angle2_deg = 30.0, 30.0   # deliberately identical angles
env_force_kN, env_angle_deg = 40.0, 200.0

a1, a2, ae = np.radians([angle1_deg, angle2_deg, env_angle_deg])

A_singular = np.array([
    [np.cos(a1), np.cos(a2)],
    [np.sin(a1), np.sin(a2)],
])
b_singular = np.array([
    -env_force_kN * np.cos(ae),
    -env_force_kN * np.sin(ae),
])

det_singular = np.linalg.det(A_singular)
print(f"det(A) with equal angles: {det_singular:.6f}")

try:
    T1, T2 = np.linalg.solve(A_singular, b_singular)
    print("Solved:", T1, T2)
except np.linalg.LinAlgError as e:
    print(f"np.linalg.solve raised LinAlgError: {e}")


det(A) with equal angles: 0.000000
Solved: 1.4448386660565178e+17 -1.4448386660565174e+17


With both lines at the identical angle, the two rows of `A` are proportional
to each other (both columns of `A` become identical), so mathematically
`det(A)` should be exactly 0. In practice it printed as `0.000000` at the
precision shown, but `np.linalg.solve` did **not** raise `LinAlgError` --
it silently returned enormous, physically meaningless tensions
(~1.4e17 kN, printed above, with T1 and T2 nearly canceling). This is an
honest and instructive real result: LAPACK's LU-based solver checks a pivot
against a numerical tolerance, and because the two angles were both
converted through `np.radians` and `np.cos`/`np.sin`, the resulting columns
are only *equal to within floating-point rounding*, not bit-for-bit
identical, so the matrix looks technically nonsingular to the solver even
though it is singular in exact arithmetic. This matters practically: relying
on `np.linalg.solve` to *raise an error* for a near-singular real system is
not safe -- checking `np.linalg.det` (or better, the condition number via
`np.linalg.cond`) before trusting a solved result is necessary, exactly as
Section 5 already recommended checking the residual.


## Homework 3 — `numpy.linalg.lstsq` on the 3-line system with only 2 equilibrium equations

The homework describes this system as "over-determined... 3 unknowns, but only
2 independent equilibrium equations." Note: 3 unknowns with only 2 equations is
actually the textbook definition of an **under-determined** system (fewer
equations than unknowns, infinitely many exact solutions), not over-determined
(which normally means *more* equations than unknowns). I implement exactly what
is described -- 3 lines, 2-D force equilibrium only, no moment equation -- and
use `lstsq` as instructed; the mismatch in the exercise's own terminology is
noted here rather than silently "fixed."


In [3]:
angles_deg_3 = np.array([30.0, 150.0, 250.0])
env_force_kN, env_angle_deg = 40.0, 200.0

a3 = np.radians(angles_deg_3)
ae = np.radians(env_angle_deg)

A_under = np.array([
    np.cos(a3),
    np.sin(a3),
])   # shape (2, 3): 2 equations, 3 unknowns
b_under = np.array([
    -env_force_kN * np.cos(ae),
    -env_force_kN * np.sin(ae),
])

print("A shape:", A_under.shape, "(2 equations, 3 unknowns -- under-determined, not over-determined)")

solution, residuals, rank, sing_vals = np.linalg.lstsq(A_under, b_under, rcond=None)
print(f"\nlstsq solution (T1, T2, T3): {solution}")
print(f"rank of A: {rank}")

check = A_under @ solution
print(f"\nA @ solution: {check}  (target b: {b_under})")


A shape: (2, 3) (2 equations, 3 unknowns -- under-determined, not over-determined)

lstsq solution (T1, T2, T3): [ 21.67474512 -16.9672914  -12.05402556]
rank of A: 2

A @ solution: [37.58770483 13.68080573]  (target b: [37.58770483 13.68080573])


**What does "best fit" mean here?** With only 2 equations for 3 unknowns,
there are infinitely many exact solutions (any point on a line in
3-tension-space satisfies both equations). `np.linalg.lstsq` does not treat
this as an approximation problem the way it would for a genuinely
over-determined system (more equations than unknowns, generally inconsistent);
instead, since `A @ solution` reproduces `b_under` above (essentially exactly,
confirming the system is exactly solvable), `lstsq` returns the
**minimum-norm** solution: among all exact solutions, the one where
`T1**2 + T2**2 + T3**2` is smallest. Physically, that is the tension
distribution that shares the load across the three lines as evenly as
possible rather than concentrating it in one line, which is a reasonable
default but is an assumption `lstsq` makes, not a real constraint anyone
supplied.


## Homework 4 — Eigenvector comparison on two uncorrelated Sonar frequency bands

Section 6 used two real, adjacent (and correlated) Sonar frequency bands
(indices 10, 11). I first search for the least-correlated pair of bands with
`np.corrcoef`, then repeat the exact same eigenvector computation on that pair.


In [4]:
import itertools
import urllib.request

sonar_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data"
raw_lines = urllib.request.urlopen(sonar_url).read().decode("utf-8").strip().split("\n")
sonar_features = np.array([[float(x) for x in line.split(",")[:-1]] for line in raw_lines])

corr_full = np.corrcoef(sonar_features.T)

best_pair, best_abs_corr = None, None
for i, j in itertools.combinations(range(sonar_features.shape[1]), 2):
    c = abs(corr_full[i, j])
    if best_abs_corr is None or c < best_abs_corr:
        best_abs_corr, best_pair = c, (i, j)

print(f"Least-correlated band pair: {best_pair}, |correlation| = {best_abs_corr:.5f}")


Least-correlated band pair: (27, 48), |correlation| = 0.00015


Repeat the covariance/eigenvector computation on this least-correlated band pair and print the dominant direction:

In [5]:
i, j = best_pair
uncorrelated_bands = sonar_features[:, [i, j]]

print("Correlation between these two real bands:", np.corrcoef(uncorrelated_bands.T)[0, 1].round(5))

cov_matrix_u = np.cov(uncorrelated_bands.T)
print("\nCovariance matrix:\n", cov_matrix_u)

eigenvalues_u, eigenvectors_u = np.linalg.eig(cov_matrix_u)
print("\nEigenvalues:", eigenvalues_u)
print("Eigenvectors (columns):\n", eigenvectors_u)

dominant_u = eigenvectors_u[:, np.argmax(eigenvalues_u)]
print(f"\nDominant direction (largest eigenvalue): {dominant_u}")


Correlation between these two real bands: -0.00015

Covariance matrix:
 [[ 5.62586375e-02 -1.30055184e-06]
 [-1.30055184e-06  1.29268796e-03]]

Eigenvalues: [0.05625864+0.j 0.00129269+0.j]
Eigenvectors (columns):
 [[ 1.00000000e+00+0.j  2.36610456e-05+0.j]
 [-2.36610456e-05+0.j  1.00000000e+00+0.j]]

Dominant direction (largest eigenvalue): [ 1.00000000e+00+0.j -2.36610456e-05+0.j]


**How does this differ from the correlated pair used in class?** For the
class's correlated pair (bands 10, 11), the dominant eigenvector points along
a clear diagonal direction (close to 45 degrees), because the two variables
move together, so most of the variance is captured along one combined
direction. For this genuinely uncorrelated pair, the covariance matrix is
close to diagonal (the off-diagonal covariance is near 0, matching the near-0
correlation printed above), so the eigenvectors come out close to the
coordinate axes themselves (roughly `[1, 0]` and `[0, 1]`, up to sign and
which eigenvalue is largest) rather than a 45-degree diagonal -- there is no
shared direction of joint variation left to rotate into, each band's variance
is essentially independent of the other's.


## Homework 5 — Broadcasting per-timestamp correction factors against Section 3's `readings`

Section 3 built a `(3, 4)` array `readings` (3 sensors, 4 timestamps) and
applied a **per-sensor** `(3,)` correction, reshaped to `(3, 1)` with
`calibration_offset[:, np.newaxis]` so it broadcasts against the 3 rows. Here
the correction is **per-timestamp** instead: a `(4,)` array, one factor per
column.


In [6]:
readings = np.array([
    [20.1, 20.3, 20.0, 20.4],   # sensor 0, 4 timestamps
    [19.8, 19.9, 20.1, 19.7],   # sensor 1
    [21.0, 21.2, 20.9, 21.1],   # sensor 2
])

timestamp_correction = np.array([1.00, 0.98, 1.02, 0.99])   # one factor per timestamp, shape (4,)

corrected_by_timestamp = readings * timestamp_correction   # no reshape needed

print("Raw readings:\n", readings)
print("\nPer-timestamp-corrected readings:\n", corrected_by_timestamp)
print("\ntimestamp_correction.shape:", timestamp_correction.shape, "-- broadcasts directly against readings.shape:", readings.shape)


Raw readings:
 [[20.1 20.3 20.  20.4]
 [19.8 19.9 20.1 19.7]
 [21.  21.2 20.9 21.1]]

Per-timestamp-corrected readings:
 [[20.1   19.894 20.4   20.196]
 [19.8   19.502 20.502 19.503]
 [21.    20.776 21.318 20.889]]

timestamp_correction.shape: (4,) -- broadcasts directly against readings.shape: (3, 4)


**What reshape does this case need, compared to the per-sensor case?**
None. NumPy broadcasting aligns shapes from the *trailing* dimension outward:
`readings` is `(3, 4)` and `timestamp_correction` is `(4,)`, and `(4,)` already
matches the trailing axis of `(3, 4)`, so it broadcasts across all 3 rows with
no reshape at all. The per-sensor case in class needed
`calibration_offset[:, np.newaxis]` specifically because a plain `(3,)` array
would try to align against the trailing axis (length 4) and fail with a shape
mismatch; reshaping to `(3, 1)` was necessary to force alignment against the
*leading* axis instead. In short: correcting along the last axis of an array
is NumPy's broadcasting default and needs no extra step; correcting along any
other axis needs an explicit `np.newaxis` to tell NumPy which axis to align.
